# Composite Evaluation — phase1_best — 2026-05-24

Checkpoint: `checkpoints/phase1/best.pth` (epoch 6, Phase 1 SCAMPS augmented training)

Evaluates on UBFC-rPPG, MCD-rPPG (IriunWebcam, held-out), rPPG-10.
Computes composite score and checks exit gate (score ≤ 1.0 = no regression from FP_PURE).

In [ ]:
# Cell 1 — Config
import sys, os, json, math, warnings, time
from pathlib import Path
from collections import OrderedDict, defaultdict

import numpy as np
import torch
import torch.nn as nn
from scipy.signal import butter, filtfilt, find_peaks
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('/mnt/sata-ssd/rppg_project')
FP_ROOT      = PROJECT_ROOT / 'external' / 'FactorizePhys'
sys.path.insert(0, str(FP_ROOT))

CHECKPOINT   = PROJECT_ROOT / 'checkpoints' / 'phase1' / 'best.pth'
UBFC_CACHE   = Path('/tmp/ubfc_y5f_clips_72.pt')
RPPG10_CACHE = PROJECT_ROOT / 'checkpoints' / 'phase1' / 'rppg10_eval_cache.pt'
MCD_CACHE    = PROJECT_ROOT / 'checkpoints' / 'phase1' / 'mcd_eval_cache.pt'

CLIP_LEN     = 160
TARGET_SIZE  = 72
HR_LO        = 40.0
HR_HI        = 200.0
FPS          = 30.0
FFT_LO       = 0.6   # Hz
FFT_HI       = 4.0   # Hz
MIN_CLIPS    = 2     # min valid clips per subject

# FP_PURE baselines
BASE_UBFC  = 1.23
BASE_MCD   = 12.36
BASE_R10   = 13.89

assert CHECKPOINT.exists(),   f'Missing: {CHECKPOINT}'
assert UBFC_CACHE.exists(),   f'Missing: {UBFC_CACHE}'
assert RPPG10_CACHE.exists(), f'Missing: {RPPG10_CACHE}'
assert MCD_CACHE.exists(),    f'Missing: {MCD_CACHE}'

print(f'Checkpoint : {CHECKPOINT}')
print(f'UBFC cache : {UBFC_CACHE} ({UBFC_CACHE.stat().st_size/1e9:.2f} GB)')
print(f'rPPG-10    : {RPPG10_CACHE}')
print(f'MCD        : {MCD_CACHE}')
print(f'CUDA devices: {torch.cuda.device_count()}')

In [ ]:
# Cell 2 — ClearML init
try:
    from clearml import Task
    task = Task.init(
        project_name='rppg',
        task_name='eval/composite/phase1_best',
        reuse_last_task_id=False,
    )
    logger = task.get_logger()
    print(f'ClearML task id: {task.id}')
except Exception as e:
    task   = None
    logger = None
    print(f'ClearML unavailable: {e}')

In [ ]:
# Cell 3 — Model loading
from yacs.config import CfgNode as CN
from neural_methods.model.FactorizePhys.FactorizePhys import FactorizePhys

def make_fp_cfg():
    cfg = CN()
    cfg.CHANNELS     = 3
    cfg.FRAME_NUM    = 160
    cfg.MD_FSAM      = True
    cfg.MD_TYPE      = 'NMF'
    cfg.MD_R         = 1
    cfg.MD_S         = 1
    cfg.MD_STEPS     = 4
    cfg.MD_RESIDUAL  = True
    cfg.MD_INFERENCE = True
    return cfg

def load_model(ckpt_path):
    device0 = torch.device('cuda:0')
    model   = FactorizePhys(frames=160, md_config=make_fp_cfg(), device=device0, in_channels=3)
    raw     = torch.load(str(ckpt_path), map_location='cpu', weights_only=False)
    sd      = OrderedDict((k.replace('module.', ''), v) for k, v in raw.items())
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if missing:     print(f'Missing keys   ({len(missing)}): {missing[:3]}...')
    if unexpected:  print(f'Unexpected keys({len(unexpected)}): {unexpected[:3]}...')
    if torch.cuda.device_count() >= 2:
        model = nn.DataParallel(model, device_ids=[0, 1])
        model = model.to(device0)
        print(f'DataParallel on GPUs 0,1')
    else:
        model = model.to(device0)
        print(f'Single GPU: cuda:0')
    model.eval()
    return model, device0

model, device = load_model(CHECKPOINT)
print(f'Model loaded from {CHECKPOINT}')

In [ ]:
# Cell 4 — Signal utilities

def extract_hr_fft(ppg, fps=FPS, lo=FFT_LO, hi=FFT_HI):
    """FFT bandpass peak HR from a 1-D waveform."""
    freqs = np.fft.rfftfreq(len(ppg), d=1.0 / fps)
    fft   = np.abs(np.fft.rfft(ppg.astype('float64')))
    mask  = (freqs >= lo) & (freqs <= hi)
    if not mask.any(): return float('nan')
    return float(freqs[mask][np.argmax(fft[mask])] * 60.0)


def extract_hr_ecg(ecg, fs=1000.0):
    """Pan-Tompkins R-peak HR from ECG. Used for rPPG-10 GT in cache."""
    nyq = fs / 2.0
    b, a = butter(4, [5.0 / nyq, 20.0 / nyq], btype='bandpass')
    filt = filtfilt(b, a, ecg.astype('float64'))
    sq   = filt ** 2
    peaks, _ = find_peaks(sq, distance=int(0.3 * fs), height=np.percentile(sq, 75))
    if len(peaks) < 2: return float('nan')
    rr = np.diff(peaks) / fs
    rr = rr[(rr >= 0.3) & (rr <= 2.0)]
    return float(60.0 / rr.mean()) if len(rr) > 0 else float('nan')


def compute_metrics(pred_hrs, gt_hrs):
    p = np.array(pred_hrs, dtype='float64')
    g = np.array(gt_hrs,   dtype='float64')
    err  = p - g
    mae  = float(np.abs(err).mean())
    rmse = float(np.sqrt((err ** 2).mean()))
    r    = float(np.corrcoef(p, g)[0, 1]) if len(p) > 1 else float('nan')
    return mae, rmse, r


@torch.no_grad()
def eval_clips(model, clips, device, gt_mode='scalar', batch=8):
    """
    gt_mode='scalar': clip['gt_hr'] is a precomputed scalar (MCD, rPPG-10)
    gt_mode='fft':    extract HR from clip['ppg'] via FFT (UBFC)
    Returns (pred_hrs_per_subj, gt_hrs_per_subj) parallel lists.
    """
    model.eval()
    subj_pred = defaultdict(list)
    subj_gt   = {}

    for i in range(0, len(clips), batch):
        b      = clips[i: i + batch]
        frames = torch.stack([c['frames'] for c in b]).to(device)
        out    = model(frames)
        preds  = out[0].float().cpu().numpy()  # (B, T)

        for j, clip in enumerate(b):
            hr = extract_hr_fft(preds[j])
            if not (HR_LO <= hr <= HR_HI):
                continue
            subj = clip['subj']
            subj_pred[subj].append(hr)
            if subj not in subj_gt:
                if gt_mode == 'scalar':
                    subj_gt[subj] = clip['gt_hr']
                else:
                    subj_gt[subj] = extract_hr_fft(clip['ppg'].numpy())

    pred_hrs, gt_hrs, valid_subjs = [], [], []
    for subj, preds in subj_pred.items():
        if len(preds) < MIN_CLIPS:
            continue
        gt = subj_gt.get(subj)
        if gt is None or (isinstance(gt, float) and math.isnan(gt)):
            continue
        pm = float(np.mean(preds))
        if not (HR_LO <= pm <= HR_HI):
            continue
        pred_hrs.append(pm)
        gt_hrs.append(gt)
        valid_subjs.append(subj)

    return pred_hrs, gt_hrs, valid_subjs


print('Signal utilities ready.')

In [ ]:
# Cell 5 — UBFC-rPPG evaluation
# GT: FFT on cached PPG waveform (30 fps)
# 42 subjects, non-overlapping 160-frame clips

print('Loading UBFC cache...')
t0 = time.time()
ubfc_clips = torch.load(str(UBFC_CACHE), weights_only=False)
print(f'UBFC clips loaded: {len(ubfc_clips)} ({time.time()-t0:.1f}s)')

print('Running UBFC inference...')
t0 = time.time()
ubfc_pred, ubfc_gt, ubfc_subjs = eval_clips(model, ubfc_clips, device, gt_mode='fft')
elapsed = time.time() - t0

ubfc_mae, ubfc_rmse, ubfc_r = compute_metrics(ubfc_pred, ubfc_gt)

print(f'UBFC results ({len(ubfc_subjs)} subjects, {elapsed:.1f}s):')
print(f'  MAE      : {ubfc_mae:.3f} bpm  (baseline {BASE_UBFC})')
print(f'  RMSE     : {ubfc_rmse:.3f} bpm')
print(f'  Pearson r: {ubfc_r:.4f}')

# Free UBFC clips from RAM
del ubfc_clips
import gc; gc.collect()

In [ ]:
# Cell 6 — MCD-rPPG held-out evaluation
# Camera: IriunWebcam only; GT: scalar pulse from db.csv (before+after averaged per subject)
# 100 held-out subjects × 2 steps × 2 clips = 400 clips

print('Loading MCD cache...')
t0 = time.time()
mcd_clips = torch.load(str(MCD_CACHE), weights_only=False)
print(f'MCD clips loaded: {len(mcd_clips)} ({time.time()-t0:.1f}s)')

print('Running MCD inference...')
t0 = time.time()
mcd_pred, mcd_gt, mcd_subjs = eval_clips(model, mcd_clips, device, gt_mode='scalar')
elapsed = time.time() - t0

mcd_mae, mcd_rmse, mcd_r = compute_metrics(mcd_pred, mcd_gt)

print(f'MCD-rPPG results ({len(mcd_subjs)} subjects, {elapsed:.1f}s):')
print(f'  MAE      : {mcd_mae:.3f} bpm  (baseline {BASE_MCD})')
print(f'  RMSE     : {mcd_rmse:.3f} bpm')
print(f'  Pearson r: {mcd_r:.4f}')

del mcd_clips; gc.collect()

In [ ]:
# Cell 7 — rPPG-10 evaluation
# All 27 subjects, Cheek1 AVI resized to 72×72, ECG GT via Pan-Tompkins
# GT is precomputed and stored in cache

print('Loading rPPG-10 cache...')
t0 = time.time()
r10_clips = torch.load(str(RPPG10_CACHE), weights_only=False)
print(f'rPPG-10 clips loaded: {len(r10_clips)} ({time.time()-t0:.1f}s)')

print('Running rPPG-10 inference...')
t0 = time.time()
r10_pred, r10_gt, r10_subjs = eval_clips(model, r10_clips, device, gt_mode='scalar')
elapsed = time.time() - t0

r10_mae, r10_rmse, r10_r = compute_metrics(r10_pred, r10_gt)

print(f'rPPG-10 results ({len(r10_subjs)} subjects, {elapsed:.1f}s):')
print(f'  MAE      : {r10_mae:.3f} bpm  (baseline {BASE_R10})')
print(f'  RMSE     : {r10_rmse:.3f} bpm')
print(f'  Pearson r: {r10_r:.4f}')

del r10_clips; gc.collect()

In [ ]:
# Cell 8 — Composite score

composite = (0.3 * (ubfc_mae / BASE_UBFC)
           + 0.3 * (mcd_mae  / BASE_MCD)
           + 0.4 * (r10_mae  / BASE_R10))

ubfc_delta = ubfc_mae - BASE_UBFC
mcd_delta  = mcd_mae  - BASE_MCD
r10_delta  = r10_mae  - BASE_R10

print('=' * 60)
print('COMPOSITE EVALUATION — phase1/best.pth')
print('=' * 60)
print(f'  UBFC  MAE : {ubfc_mae:.3f} bpm  (baseline {BASE_UBFC:.2f}, delta {ubfc_delta:+.3f})')
print(f'  MCD   MAE : {mcd_mae:.3f} bpm  (baseline {BASE_MCD:.2f}, delta {mcd_delta:+.3f})')
print(f'  rPPG10 MAE: {r10_mae:.3f} bpm  (baseline {BASE_R10:.2f}, delta {r10_delta:+.3f})')
print(f'  Composite : {composite:.4f}  (baseline = 1.0000)')
print()

if composite <= 1.0:
    print('EXIT GATE: PASSED  (score <= 1.0 — improvement or no regression from FP_PURE)')
    print('Phase 2 starting point: checkpoints/phase1/best.pth')
    verdict = 'improvement' if composite < 0.999 else 'neutral'
else:
    print('EXIT GATE: FAILED  (score > 1.0 — regression from FP_PURE)')
    print('Phase 2 starting point: checkpoints/best_pretrained/best.pth (FP_PURE)')
    verdict = 'regression'

print(f'Verdict: {verdict}')

In [ ]:
# Cell 9 — ClearML logging
if task is not None and logger is not None:
    logger.report_scalar('MAE', 'UBFC',    value=ubfc_mae,  iteration=0)
    logger.report_scalar('MAE', 'MCD',     value=mcd_mae,   iteration=0)
    logger.report_scalar('MAE', 'rPPG-10', value=r10_mae,   iteration=0)
    logger.report_scalar('RMSE', 'UBFC',   value=ubfc_rmse, iteration=0)
    logger.report_scalar('RMSE', 'MCD',    value=mcd_rmse,  iteration=0)
    logger.report_scalar('RMSE', 'rPPG-10',value=r10_rmse,  iteration=0)
    logger.report_scalar('Pearson_r', 'UBFC',    value=ubfc_r, iteration=0)
    logger.report_scalar('Pearson_r', 'MCD',     value=mcd_r,  iteration=0)
    logger.report_scalar('Pearson_r', 'rPPG-10', value=r10_r,  iteration=0)
    logger.report_scalar('Composite', 'score',   value=composite, iteration=0)
    task.upload_artifact('results', artifact_object={
        'ubfc_mae': ubfc_mae, 'ubfc_rmse': ubfc_rmse, 'ubfc_r': ubfc_r,
        'mcd_mae':  mcd_mae,  'mcd_rmse':  mcd_rmse,  'mcd_r':  mcd_r,
        'r10_mae':  r10_mae,  'r10_rmse':  r10_rmse,  'r10_r':  r10_r,
        'composite': composite, 'verdict': verdict,
        'checkpoint': str(CHECKPOINT),
    })
    task.close()
    print(f'ClearML logged: rppg / eval/composite/phase1_best')
else:
    print('ClearML not available — results not logged.')

In [ ]:
# Cell 10 — Final report
print()
print('╔' + '═'*58 + '╗')
print('║  COMPOSITE EVALUATION REPORT — phase1/best.pth          ║')
print('╠' + '═'*58 + '╣')
print(f'║  Dataset     │  MAE (bpm)  │  RMSE (bpm) │  Pearson r  ║')
print(f'║  UBFC        │  {ubfc_mae:6.3f}     │  {ubfc_rmse:6.3f}     │  {ubfc_r:7.4f}    ║')
print(f'║  MCD-rPPG    │  {mcd_mae:6.3f}     │  {mcd_rmse:6.3f}     │  {mcd_r:7.4f}    ║')
print(f'║  rPPG-10     │  {r10_mae:6.3f}     │  {r10_rmse:6.3f}     │  {r10_r:7.4f}    ║')
print('╠' + '═'*58 + '╣')
print(f'║  Composite score : {composite:.4f}  (baseline = 1.0000)       ║')
print(f'║  Verdict         : {verdict:<38}  ║')
print('╚' + '═'*58 + '╝')
print()
print('Baselines (FP_PURE):')
print(f'  UBFC   1.23 bpm → {ubfc_mae:.3f} ({ubfc_delta:+.3f})')
print(f'  MCD   12.36 bpm → {mcd_mae:.3f} ({mcd_delta:+.3f})')
print(f'  rPPG10 13.89 bpm → {r10_mae:.3f} ({r10_delta:+.3f})')
print()
if composite > 1.0:
    regressions = []
    if ubfc_delta > 0: regressions.append(f'UBFC +{ubfc_delta:.3f}')
    if mcd_delta  > 0: regressions.append(f'MCD +{mcd_delta:.3f}')
    if r10_delta  > 0: regressions.append(f'rPPG-10 +{r10_delta:.3f}')
    print(f'Regressions: {" | ".join(regressions)}')
    print('Phase 2 starts from: checkpoints/best_pretrained/best.pth (FP_PURE)')
else:
    print('Phase 2 starts from: checkpoints/phase1/best.pth')